# FLUX inference schedule comparison without retraining

This Modal notebook tests whether the saved 12-epoch overfitting checkpoint is
being underestimated by the original 50-step linear Euler sampler.

Success criteria:

- compare linear and official BFL-shifted timestep schedules with identical noise;
- report valid-grid, shape, cell-accuracy, and exact-grid results;
- inspect both VAE RGB outputs and strictly decoded ARC grids inline;
- keep model loading separate so the experiment and display cells can be rerun;
- perform no training and never modify model weights.


In [ ]:
!nvidia-smi

In [ ]:
%uv pip uninstall torchvision

## 1. Fixed run and checkpoint configuration

The checkpoint path is explicit because the training report failed during Modal
Volume finalization and therefore has no saved_checkpoint entry. The validation
below accepts only the known completed 12-epoch run and inspects the standalone
transformer files directly.


In [ ]:
from __future__ import annotations

import base64
from dataclasses import asdict, replace
from datetime import datetime, timezone
import hashlib
import importlib
import importlib.metadata
import json
import math
import os
from pathlib import Path
import platform
import shutil
import subprocess
import sys
import time

TRAINING_RUN_ID = '20260825T150620231361Z'
CHECKPOINT_DIRECTORY_NAME = (
    '.flux-transformer-step-2496.partial-c9c70c21fb254c55a9f6fcf4544df352'
)
SOURCE_REVISION = '58c31c495be8c7fa04cc9ee29b491ba175568e81'
EXPECTED_GLOBAL_STEP = 2496
EXPECTED_COMPLETED_EPOCHS = 12
EXPECTED_TASKS = 8
MINIMUM_VRAM_GIB = 70.0
REPOSITORY_URL = 'https://github.com/joshuakauling/diffusion-arc.git'

VOLUME_ROOT = Path('/mnt/diffusion-arc-models')
BASE_FLUX_CHECKPOINT = VOLUME_ROOT / 'flux2-klein-base-4b'
NVARC_CHECKPOINT = VOLUME_ROOT / 'nvarc-qwen3-4b-grids15-sft139'
MODEL_MANIFEST = VOLUME_ROOT / 'download-manifest.json'
ARC_DATA_ROOT = VOLUME_ROOT / 'data' / 'raw' / 'arc-agi-2'
ARC_DATA_MANIFEST = VOLUME_ROOT / 'data' / 'arc-agi-2-manifest.json'
TRAINING_RUN_ROOT = (
    VOLUME_ROOT
    / 'results'
    / 'single-gpu-overfitting-pilot'
    / TRAINING_RUN_ID
)
TRAINING_REPORT_PATH = TRAINING_RUN_ROOT / 'report.json'
TRANSFORMER_CHECKPOINT = TRAINING_RUN_ROOT / CHECKPOINT_DIRECTORY_NAME

required_paths = (
    BASE_FLUX_CHECKPOINT,
    NVARC_CHECKPOINT,
    MODEL_MANIFEST,
    ARC_DATA_ROOT,
    ARC_DATA_MANIFEST,
    TRAINING_REPORT_PATH,
    TRANSFORMER_CHECKPOINT,
)
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(f'Missing required Volume artifacts: {missing_paths}')
if not TRANSFORMER_CHECKPOINT.is_dir():
    raise NotADirectoryError(TRANSFORMER_CHECKPOINT)

EVALUATION_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT_ROOT = (
    TRAINING_RUN_ROOT / 'inference-schedule-comparisons' / EVALUATION_ID
)


def run_command(
    label: str,
    command: list[str],
    *,
    cwd: Path | None = None,
    env: dict[str, str] | None = None,
) -> str:
    completed = subprocess.run(
        command,
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )
    if completed.returncode != 0:
        tail = '\n'.join(completed.stdout.splitlines()[-40:])
        if tail:
            print(tail)
        raise RuntimeError(f'{label} failed with exit code {completed.returncode}')
    return completed.stdout.strip()


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


print('Training report:', TRAINING_REPORT_PATH)
print('Transformer checkpoint:', TRANSFORMER_CHECKPOINT)
print('Optional output directory:', OUTPUT_ROOT)


## 2. Fetch and test the exact source revision

Run this setup cell once. Reexecuting it in the same kernel reuses the verified
source checkout. A GitHub token must be attached through a Modal Secret.


In [ ]:
SOURCE_DIR = Path('/tmp') / f'diffusion-arc-schedule-eval-{EVALUATION_ID}'
if not SOURCE_DIR.exists():
    SOURCE_DIR.mkdir(parents=True)
    run_command('git init', ['git', 'init'], cwd=SOURCE_DIR)
    run_command(
        'add origin',
        ['git', 'remote', 'add', 'origin', REPOSITORY_URL],
        cwd=SOURCE_DIR,
    )

    github_token = os.environ.get('GITHUB_TOKEN', '').strip()
    if not github_token:
        raise RuntimeError('Attach a Modal Secret containing GITHUB_TOKEN')
    credential = base64.b64encode(
        f'x-access-token:{github_token}'.encode('utf-8')
    ).decode('ascii')
    git_environment = os.environ.copy()
    git_environment.pop('GITHUB_TOKEN', None)
    git_environment.update({
        'GIT_TERMINAL_PROMPT': '0',
        'GIT_CONFIG_COUNT': '1',
        'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader',
        'GIT_CONFIG_VALUE_0': f'AUTHORIZATION: basic {credential}',
    })
    try:
        run_command(
            'fetch source',
            ['git', 'fetch', '--depth=1', 'origin', SOURCE_REVISION],
            cwd=SOURCE_DIR,
            env=git_environment,
        )
        run_command(
            'checkout source',
            ['git', 'checkout', '--detach', 'FETCH_HEAD'],
            cwd=SOURCE_DIR,
        )
    finally:
        git_environment.clear()
        del git_environment
        del credential

RESOLVED_SOURCE_COMMIT = run_command(
    'resolve source', ['git', 'rev-parse', 'HEAD'], cwd=SOURCE_DIR
)
if RESOLVED_SOURCE_COMMIT != SOURCE_REVISION:
    raise RuntimeError(
        f'Resolved source {RESOLVED_SOURCE_COMMIT} differs from {SOURCE_REVISION}'
    )

PINNED_PACKAGES = {
    'accelerate': '1.12.0',
    'diffusers': '0.39.0',
    'einops': '0.8.1',
    'matplotlib': '3.10.6',
    'Pillow': '11.3.0',
    'safetensors': '0.8.0',
    'torch': '2.13.0',
    'tqdm': '4.67.1',
    'transformers': '4.56.1',
}


def installed_version(name: str) -> str | None:
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


requirements = [
    f'{name}=={version}'
    for name, version in PINNED_PACKAGES.items()
    if installed_version(name) != version
]
if requirements:
    run_command(
        'install pinned dependencies',
        [
            sys.executable,
            '-m',
            'pip',
            'install',
            '--quiet',
            '--no-input',
            '--no-cache-dir',
            *requirements,
        ],
    )
run_command(
    'install Diffusion-ARC source',
    [
        sys.executable,
        '-m',
        'pip',
        'install',
        '--quiet',
        '--no-input',
        '--no-deps',
        '--no-build-isolation',
        '-e',
        str(SOURCE_DIR),
    ],
)

source_path = str((SOURCE_DIR / 'src').resolve())
if source_path not in sys.path:
    sys.path.insert(0, source_path)
importlib.invalidate_caches()
import diffusion_arc

test_environment = os.environ.copy()
test_environment['PYTHONDONTWRITEBYTECODE'] = '1'
test_environment['PYTHONPATH'] = source_path
test_started = time.perf_counter()
test_process = subprocess.run(
    [sys.executable, '-B', '-m', 'unittest', 'discover', '-s', 'tests'],
    cwd=SOURCE_DIR,
    env=test_environment,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    check=False,
)
print('\n'.join(test_process.stdout.splitlines()[-30:]))
if test_process.returncode != 0:
    raise RuntimeError(f'Unit tests returned {test_process.returncode}')
print('Evaluation source:', RESOLVED_SOURCE_COMMIT)
print(f'Unit tests: PASS in {time.perf_counter() - test_started:.2f}s')


## 3. Validate GPU, run provenance, dataset, split, and checkpoint

The failed training status is accepted only when all recorded failures belong
to checkpoint finalization after 12 completed epochs. The Safetensors index and
every shard header must agree before any model is loaded.


In [ ]:
import torch
from safetensors import safe_open

if not torch.cuda.is_available():
    raise RuntimeError('CUDA is unavailable')
if not torch.cuda.is_bf16_supported():
    raise RuntimeError('Selected CUDA device does not support BF16')
DEVICE = torch.device('cuda:0')
torch.cuda.set_device(DEVICE)
properties = torch.cuda.get_device_properties(DEVICE)
gpu_vram_gib = properties.total_memory / 2**30
if gpu_vram_gib < MINIMUM_VRAM_GIB:
    raise RuntimeError(
        f'Requires at least {MINIMUM_VRAM_GIB:.0f} GiB VRAM; '
        f'received {properties.name} with {gpu_vram_gib:.1f} GiB'
    )


def cuda_memory(label: str) -> dict[str, object]:
    torch.cuda.synchronize(DEVICE)
    return {
        'label': label,
        'allocated_gib': torch.cuda.memory_allocated(DEVICE) / 2**30,
        'reserved_gib': torch.cuda.memory_reserved(DEVICE) / 2**30,
        'peak_allocated_gib': torch.cuda.max_memory_allocated(DEVICE) / 2**30,
        'peak_reserved_gib': torch.cuda.max_memory_reserved(DEVICE) / 2**30,
    }


training_report = json.loads(TRAINING_REPORT_PATH.read_text(encoding='utf-8'))
if training_report.get('completed_optimizer_steps') != EXPECTED_GLOBAL_STEP:
    raise RuntimeError('Training report has an unexpected optimizer-step count')
if training_report.get('completed_epochs') != EXPECTED_COMPLETED_EPOCHS:
    raise RuntimeError('Training report has an unexpected completed-epoch count')
if training_report.get('status') == 'FAIL':
    failures = training_report.get('failures')
    if not isinstance(failures, list) or not failures:
        raise RuntimeError('Failed training report contains no failure records')
    if any(item.get('stage') != 'checkpoint-finalization' for item in failures):
        raise RuntimeError('Training failed outside checkpoint finalization')
    print('Accepted known report failure: checkpoint finalization only')
elif training_report.get('status') != 'COMPLETE':
    raise RuntimeError(f'Unexpected training status: {training_report.get("status")!r}')

if training_report['source']['resolved_commit'] != SOURCE_REVISION:
    raise RuntimeError('Evaluation source differs from the training source')
if sha256_file(MODEL_MANIFEST) != training_report['model_manifest_sha256']:
    raise RuntimeError('Model manifest differs from the training run')
if sha256_file(ARC_DATA_MANIFEST) != training_report['dataset']['manifest_sha256']:
    raise RuntimeError('Dataset manifest differs from the training run')

model_manifest = json.loads(MODEL_MANIFEST.read_text(encoding='utf-8'))
if Path(model_manifest['flux']['path']).resolve() != BASE_FLUX_CHECKPOINT.resolve():
    raise RuntimeError('FLUX base checkpoint differs from the model manifest')
if Path(model_manifest['nvarc']['path']).resolve() != NVARC_CHECKPOINT.resolve():
    raise RuntimeError('NVARC checkpoint differs from the model manifest')

dataset_manifest = json.loads(ARC_DATA_MANIFEST.read_text(encoding='utf-8'))
for task_id in training_report['dataset']['task_ids']:
    relative = f'training/{task_id}.json'
    task_path = ARC_DATA_ROOT / relative
    expected_hash = dataset_manifest['files'].get(relative)
    if expected_hash is None or not task_path.is_file():
        raise RuntimeError(f'Dataset manifest lacks selected task: {relative}')
    if sha256_file(task_path) != expected_hash:
        raise RuntimeError(f'Selected raw task differs: {relative}')

TEMP_DATA_ROOT = Path('/tmp') / f'diffusion-arc-schedule-data-{EVALUATION_ID}' / 'data'
RUNTIME_SPLIT_ROOT = TEMP_DATA_ROOT / 'splits' / 'overfitting_run_1'
if not RUNTIME_SPLIT_ROOT.exists():
    RUNTIME_SPLIT_ROOT.parent.mkdir(parents=True)
    shutil.copytree(
        SOURCE_DIR / 'data' / 'splits' / 'overfitting_run_1',
        RUNTIME_SPLIT_ROOT,
    )
raw_root = TEMP_DATA_ROOT / 'raw'
raw_root.mkdir(exist_ok=True)
raw_link = raw_root / 'arc-agi-2'
if not raw_link.exists():
    os.symlink(ARC_DATA_ROOT, raw_link, target_is_directory=True)

actual_split_hashes = {
    str(path.relative_to(RUNTIME_SPLIT_ROOT)): sha256_file(path)
    for path in sorted(RUNTIME_SPLIT_ROOT.rglob('*.json'))
}
if actual_split_hashes != training_report['dataset']['split_file_sha256']:
    raise RuntimeError('Evaluation split files differ from the training run')


def validate_transformer_checkpoint(directory: Path) -> dict[str, object]:
    config_path = directory / 'config.json'
    if not config_path.is_file():
        raise FileNotFoundError(f'Missing transformer config: {config_path}')
    config = json.loads(config_path.read_text(encoding='utf-8'))
    if not isinstance(config, dict) or not config:
        raise RuntimeError('Transformer config is empty or invalid')

    index_paths = sorted(directory.glob('*.safetensors.index.json'))
    if len(index_paths) != 1:
        raise RuntimeError(
            f'Expected exactly one Safetensors index, found {len(index_paths)}'
        )
    index_path = index_paths[0]
    index = json.loads(index_path.read_text(encoding='utf-8'))
    weight_map = index.get('weight_map')
    if not isinstance(weight_map, dict) or not weight_map:
        raise RuntimeError('Safetensors index has no weight_map')

    shard_names = sorted(set(weight_map.values()))
    if any(not isinstance(name, str) or Path(name).name != name for name in shard_names):
        raise RuntimeError('Safetensors index contains an unsafe shard path')
    missing_shards = [name for name in shard_names if not (directory / name).is_file()]
    if missing_shards:
        raise FileNotFoundError(f'Missing checkpoint shards: {missing_shards}')

    tensors_by_shard = {
        shard_name: {name for name, mapped in weight_map.items() if mapped == shard_name}
        for shard_name in shard_names
    }
    total_bytes = 0
    for shard_name in shard_names:
        shard_path = directory / shard_name
        if shard_path.stat().st_size <= 0:
            raise RuntimeError(f'Checkpoint shard is empty: {shard_name}')
        total_bytes += shard_path.stat().st_size
        with safe_open(shard_path, framework='pt', device='cpu') as handle:
            actual_keys = set(handle.keys())
        if actual_keys != tensors_by_shard[shard_name]:
            raise RuntimeError(f'Shard tensor map differs: {shard_name}')

    expected_names = {'config.json', index_path.name, *shard_names}
    extra_files = sorted(
        path.name for path in directory.iterdir()
        if path.is_file() and path.name not in expected_names
    )
    return {
        'path': str(directory),
        'index': index_path.name,
        'shards': shard_names,
        'tensor_count': len(weight_map),
        'total_bytes': total_bytes,
        'extra_files': extra_files,
    }


CHECKPOINT_MANIFEST = validate_transformer_checkpoint(TRANSFORMER_CHECKPOINT)

from diffusion_arc.training.single_gpu_config import load_single_gpu_training_config

source_config = load_single_gpu_training_config(
    SOURCE_DIR / 'configs' / 'training' / 'single_gpu_overfitting_pilot_v2.json'
)
RUN_CONFIG = replace(
    source_config,
    models=replace(
        source_config.models,
        flux_checkpoint=BASE_FLUX_CHECKPOINT,
        nvarc_checkpoint=NVARC_CHECKPOINT,
    ),
    data=replace(source_config.data, split_directory=RUNTIME_SPLIT_ROOT),
)
TASK_IDS = tuple(training_report['dataset']['task_ids'])
if len(TASK_IDS) != EXPECTED_TASKS:
    raise RuntimeError(f'Expected {EXPECTED_TASKS} tasks, received {len(TASK_IDS)}')

print(f'GPU: {properties.name} | {gpu_vram_gib:.1f} GiB')
print(
    f'Checkpoint: {len(CHECKPOINT_MANIFEST["shards"])} shards, '
    f'{CHECKPOINT_MANIFEST["tensor_count"]} tensors, '
    f'{CHECKPOINT_MANIFEST["total_bytes"] / 2**30:.2f} GiB'
)
if CHECKPOINT_MANIFEST['extra_files']:
    print('Ignored non-model files:', CHECKPOINT_MANIFEST['extra_files'])


## 4. Load the frozen inference stack once

NVARC, the VAE, and the saved transformer are frozen. No optimizer is created.
Do not rerun this cell when changing only the lower interactive configuration.


In [ ]:
from diffusion_arc.training.single_gpu_setup import (
    load_single_gpu_models,
    prepare_single_gpu_device,
    seed_single_gpu_process,
)

os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

device = prepare_single_gpu_device(RUN_CONFIG)
seed_single_gpu_process(167)
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats(device)

load_started = time.perf_counter()
models = load_single_gpu_models(
    RUN_CONFIG,
    device=device,
    transformer_checkpoint=TRANSFORMER_CHECKPOINT,
)
models.transformer.requires_grad_(False)
models.transformer.eval()
load_seconds = time.perf_counter() - load_started

if any(parameter.requires_grad for parameter in models.nvarc_runner.parameters()):
    raise RuntimeError('NVARC unexpectedly trainable')
if any(parameter.requires_grad for parameter in models.vae_decoder.parameters()):
    raise RuntimeError('VAE unexpectedly trainable')
if any(parameter.requires_grad for parameter in models.transformer.parameters()):
    raise RuntimeError('Evaluation transformer unexpectedly trainable')

MODEL_LOAD_SUMMARY = {
    'load_seconds': load_seconds,
    'transformer_parameters': sum(
        parameter.numel() for parameter in models.transformer.parameters()
    ),
    'memory': cuda_memory('after-model-load'),
}
MODEL_LOAD_SUMMARY


## 5. Reconstruct the exact V2 evaluation panels

The keys come from the training report. For every selected key, the target-free
evaluation tokens must equal the exact prefix preceding the final training
target. NVARC conditioning is cached lazily per panel and task selection.


In [ ]:
from diffusion_arc.arc_data import (
    ArcEvaluationTask,
    ArcQuery,
    apply_training_augmentation,
    load_training_task,
    render_evaluation_prompts,
    tokenize_nvarc_evaluation_sample,
)
from diffusion_arc.inference import prepare_evaluation_text_inputs
from diffusion_arc.training import ArcTrainingDataset

PANEL_VIEWS = {
    'seen_training_augmentation': 'train',
    'canonical_original': 'train_original',
    'unseen_augmentation': 'train_unseen',
}


def build_panel_prompts(panel_name: str) -> dict[str, object]:
    view_name = PANEL_VIEWS[panel_name]
    dataset = ArcTrainingDataset(
        split_directory=RUNTIME_SPLIT_ROOT,
        view_name=view_name,
        tokenizer=models.tokenizer,
    )
    keys_to_indices = {str(key): index for index, key in enumerate(dataset.keys)}
    expected_entry = training_report['evaluation_panels'][panel_name]
    expected_keys = tuple(expected_entry['keys'])

    samples = []
    expected_grids = []
    task_ids = []
    token_counts = []
    for serialized_key in expected_keys:
        if serialized_key not in keys_to_indices:
            raise RuntimeError(f'Reported key is absent from {view_name}: {serialized_key}')
        index = keys_to_indices[serialized_key]
        key = dataset.keys[index]
        training_sample = dataset[index]
        raw_task = load_training_task(dataset.source_directory / f'{key.task_id}.json')
        augmented = apply_training_augmentation(raw_task, key.augmentation)
        query_pair = augmented.pairs[-1]
        evaluation_task = ArcEvaluationTask(
            task_id=serialized_key,
            demonstrations=augmented.pairs[:-1],
            queries=(ArcQuery(query_index=0, input_grid=query_pair.input_grid),),
        )
        rendered = render_evaluation_prompts(evaluation_task)[0]
        tokenized = tokenize_nvarc_evaluation_sample(
            rendered,
            models.tokenizer,
            max_tokens=dataset.max_tokens,
        )
        training_prefix = training_sample.input_ids[:training_sample.prefix_token_ends[-1]]
        if tokenized.input_ids != training_prefix:
            raise RuntimeError(f'Evaluation tokens differ from training prefix: {key}')
        if training_sample.target_grids[-1] != query_pair.output_grid:
            raise RuntimeError(f'Expected target differs from training sample: {key}')

        samples.append(tokenized)
        expected_grids.append(query_pair.output_grid)
        task_ids.append(key.task_id)
        token_counts.append(len(tokenized.input_ids))

    if tuple(task_ids) != TASK_IDS:
        raise RuntimeError(f'{panel_name} task order differs from the training report')
    if tuple(token_counts) != tuple(expected_entry['target_free_token_counts']):
        raise RuntimeError(f'{panel_name} token counts differ from the training report')
    return {
        'panel_name': panel_name,
        'view_name': view_name,
        'keys': expected_keys,
        'task_ids': tuple(task_ids),
        'samples': tuple(samples),
        'expected_grids': tuple(expected_grids),
        'token_counts': tuple(token_counts),
    }


PANEL_PROMPTS = {
    name: build_panel_prompts(name)
    for name in PANEL_VIEWS
}
CONDITIONING_CACHE: dict[tuple[str, tuple[str, ...]], dict[str, object]] = {}


def get_conditioned_panel(
    panel_name: str,
    selected_task_ids: tuple[str, ...],
) -> dict[str, object]:
    if panel_name not in PANEL_PROMPTS:
        raise ValueError(f'Unknown panel: {panel_name}')
    panel = PANEL_PROMPTS[panel_name]
    available = tuple(panel['task_ids'])
    if selected_task_ids:
        unknown = tuple(task_id for task_id in selected_task_ids if task_id not in available)
        if unknown:
            raise ValueError(f'Unknown task IDs for {panel_name}: {unknown}')
        chosen = tuple(task_id for task_id in available if task_id in selected_task_ids)
    else:
        chosen = available
    cache_key = (panel_name, chosen)
    if cache_key in CONDITIONING_CACHE:
        return CONDITIONING_CACHE[cache_key]

    indices = tuple(available.index(task_id) for task_id in chosen)
    conditioned = {
        'panel_name': panel_name,
        'task_ids': chosen,
        'keys': tuple(panel['keys'][index] for index in indices),
        'expected_grids': tuple(panel['expected_grids'][index] for index in indices),
        'text_inputs': prepare_evaluation_text_inputs(
            tuple(panel['samples'][index] for index in indices),
            nvarc_runner=models.nvarc_runner,
        ),
    }
    CONDITIONING_CACHE[cache_key] = conditioned
    return conditioned


{name: panel['keys'] for name, panel in PANEL_PROMPTS.items()}


## 6. Define and inspect the sampler schedules

The BFL equations match the released FLUX.2 sampler constants used by the
project before the inference API was simplified. Both schedules still use the
same backward Euler implementation; only the timestep boundaries differ.


In [ ]:
from diffusion_arc.inference import linear_timestep_schedule
from diffusion_arc.model.contracts import TARGET_LATENT_SHAPE

IMAGE_SEQUENCE_LENGTH = TARGET_LATENT_SHAPE[0]


def compute_empirical_mu(image_seq_len: int, num_steps: int) -> float:
    if isinstance(image_seq_len, bool) or not isinstance(image_seq_len, int):
        raise TypeError('image_seq_len must be an integer')
    if isinstance(num_steps, bool) or not isinstance(num_steps, int):
        raise TypeError('num_steps must be an integer')
    if image_seq_len <= 0 or num_steps <= 0:
        raise ValueError('image_seq_len and num_steps must be positive')
    a1, b1 = 8.73809524e-05, 1.89833333
    a2, b2 = 0.00016927, 0.45666666
    if image_seq_len > 4300:
        return float(a2 * image_seq_len + b2)
    mu_200 = a2 * image_seq_len + b2
    mu_10 = a1 * image_seq_len + b1
    slope = (mu_200 - mu_10) / 190.0
    intercept = mu_200 - 200.0 * slope
    return float(slope * num_steps + intercept)


def bfl_timestep_schedule(
    num_steps: int,
    *,
    image_seq_len: int = IMAGE_SEQUENCE_LENGTH,
    device: torch.device | None = None,
) -> torch.Tensor:
    mu = compute_empirical_mu(image_seq_len, num_steps)
    timesteps = torch.linspace(
        1.0,
        0.0,
        steps=num_steps + 1,
        device=device,
        dtype=torch.float32,
    )
    exp_mu = math.exp(mu)
    shifted = exp_mu / (exp_mu + (1.0 / timesteps - 1.0))
    shifted[0] = 1.0
    shifted[-1] = 0.0
    if not bool(torch.isfinite(shifted).all().item()):
        raise RuntimeError('BFL schedule contains non-finite values')
    if not bool((shifted[:-1] > shifted[1:]).all().item()):
        raise RuntimeError('BFL schedule is not strictly decreasing')
    return shifted


def build_schedule(
    schedule_name: str,
    num_steps: int,
    *,
    device: torch.device | None,
) -> torch.Tensor:
    if schedule_name == 'linear':
        schedule = linear_timestep_schedule(num_steps, device=device)
    elif schedule_name == 'bfl':
        schedule = bfl_timestep_schedule(num_steps, device=device)
    else:
        raise ValueError(f'Unknown schedule: {schedule_name}')
    if schedule.shape != (num_steps + 1,):
        raise RuntimeError('Schedule has an unexpected shape')
    if schedule[0].item() != 1.0 or schedule[-1].item() != 0.0:
        raise RuntimeError('Schedule endpoints differ from 1 and 0')
    return schedule


for schedule_name in ('linear', 'bfl'):
    smoke_schedule = build_schedule(schedule_name, 50, device=None)
    assert smoke_schedule.shape == (51,)
print('Schedule smoke checks: PASS | image sequence length:', IMAGE_SEQUENCE_LENGTH)


## 7. Interactive comparison configuration

Edit and rerun this cell, then rerun the schedule preview and inference cells.
An empty ACTIVE_TASK_IDS tuple selects all eight tasks. Start with the seen panel;
add the other panels once the most promising sampler setting is known.


In [ ]:
# Safe quick start: one panel, all tasks, two matched seeds.
ACTIVE_PANEL_NAMES = ('seen_training_augmentation',)
ACTIVE_TASK_IDS: tuple[str, ...] = ()
ACTIVE_SCHEDULE_SPECS = (
    ('linear', 50),
    ('bfl', 50),
    ('bfl', 100),
    ('bfl', 200),
)
ACTIVE_GENERATION_SEEDS = (167, 168)

# False reuses completed runs from the in-memory cache.
FORCE_RERUN = False

# Saving writes only JSON and PNG outputs; it never calls modal.Volume.commit().
SAVE_RESULTS = True

if not ACTIVE_PANEL_NAMES or len(set(ACTIVE_PANEL_NAMES)) != len(ACTIVE_PANEL_NAMES):
    raise ValueError('ACTIVE_PANEL_NAMES must be non-empty and unique')
if any(name not in PANEL_VIEWS for name in ACTIVE_PANEL_NAMES):
    raise ValueError(f'Unknown panel selection: {ACTIVE_PANEL_NAMES}')
if len(set(ACTIVE_TASK_IDS)) != len(ACTIVE_TASK_IDS):
    raise ValueError('ACTIVE_TASK_IDS must be unique')
if not ACTIVE_SCHEDULE_SPECS or len(set(ACTIVE_SCHEDULE_SPECS)) != len(ACTIVE_SCHEDULE_SPECS):
    raise ValueError('ACTIVE_SCHEDULE_SPECS must be non-empty and unique')
for name, steps in ACTIVE_SCHEDULE_SPECS:
    build_schedule(name, steps, device=None)
if not ACTIVE_GENERATION_SEEDS or len(set(ACTIVE_GENERATION_SEEDS)) != len(ACTIVE_GENERATION_SEEDS):
    raise ValueError('ACTIVE_GENERATION_SEEDS must be non-empty and unique')

estimated_step_units = (
    len(ACTIVE_PANEL_NAMES)
    * len(ACTIVE_GENERATION_SEEDS)
    * sum(steps for _, steps in ACTIVE_SCHEDULE_SPECS)
)
print('Panels:', ACTIVE_PANEL_NAMES)
print('Tasks:', ACTIVE_TASK_IDS or TASK_IDS)
print('Schedules:', ACTIVE_SCHEDULE_SPECS)
print('Seeds:', ACTIVE_GENERATION_SEEDS)
print('Total transformer step units:', estimated_step_units)


In [ ]:
import matplotlib.pyplot as plt

figure, axis = plt.subplots(figsize=(8, 5))
for schedule_name, num_steps in ACTIVE_SCHEDULE_SPECS:
    values = build_schedule(schedule_name, num_steps, device=None).cpu()
    fraction = torch.linspace(0.0, 1.0, values.numel())
    label = f'{schedule_name} / {num_steps} steps'
    axis.plot(fraction.numpy(), values.numpy(), label=label)
axis.set_title(f'Timestep schedules for {IMAGE_SEQUENCE_LENGTH} image tokens')
axis.set_xlabel('Euler progress')
axis.set_ylabel('timestep')
axis.grid(alpha=0.25)
axis.legend()
plt.show()


## 8. Run or extend the inference cache

Every schedule receives a fresh generator initialized with the same seed, so
the initial latent noise is identical within each panel and seed. The progress
bar advances per complete panel/schedule/seed generation.


In [ ]:
from tqdm.auto import tqdm

from diffusion_arc.inference import diagnose_exact_grids, generate_arc_batch

if 'INFERENCE_CACHE' not in globals():
    INFERENCE_CACHE: dict[tuple[object, ...], dict[str, object]] = {}


def inference_cache_key(
    panel_name: str,
    task_ids: tuple[str, ...],
    schedule_name: str,
    num_steps: int,
    seed: int,
) -> tuple[object, ...]:
    return (
        str(TRANSFORMER_CHECKPOINT),
        panel_name,
        task_ids,
        schedule_name,
        num_steps,
        seed,
    )


jobs = [
    (panel_name, schedule_name, num_steps, seed)
    for panel_name in ACTIVE_PANEL_NAMES
    for schedule_name, num_steps in ACTIVE_SCHEDULE_SPECS
    for seed in ACTIVE_GENERATION_SEEDS
]
progress = tqdm(jobs, desc='Schedule inference', unit='generation')
for panel_name, schedule_name, num_steps, seed in progress:
    conditioned = get_conditioned_panel(panel_name, ACTIVE_TASK_IDS)
    chosen_task_ids = tuple(conditioned['task_ids'])
    cache_key = inference_cache_key(
        panel_name,
        chosen_task_ids,
        schedule_name,
        num_steps,
        seed,
    )
    progress.set_postfix_str(f'{panel_name} | {schedule_name}-{num_steps} | seed {seed}')
    if cache_key in INFERENCE_CACHE and not FORCE_RERUN:
        continue

    schedule = build_schedule(schedule_name, num_steps, device=device)
    torch.cuda.reset_peak_memory_stats(device)
    started = time.perf_counter()
    generated = generate_arc_batch(
        models.transformer,
        models.vae_decoder,
        conditioned['text_inputs'],
        timestep_schedule=schedule,
        compute_dtype=models.flux_compute_dtype,
        generator=torch.Generator(device=device).manual_seed(seed),
    )
    seconds = time.perf_counter() - started
    diagnostics = diagnose_exact_grids(
        generated.predictions,
        conditioned['expected_grids'],
    )
    INFERENCE_CACHE[cache_key] = {
        'panel_name': panel_name,
        'task_ids': chosen_task_ids,
        'keys': tuple(conditioned['keys']),
        'expected_grids': tuple(conditioned['expected_grids']),
        'schedule_name': schedule_name,
        'num_steps': num_steps,
        'seed': seed,
        'seconds': seconds,
        'schedule': generated.timestep_schedule,
        'rgb_images': generated.rgb_images,
        'class_canvases': generated.class_canvases,
        'predictions': generated.predictions,
        'diagnostics': diagnostics,
        'memory': cuda_memory(
            f'{panel_name}-{schedule_name}-{num_steps}-seed-{seed}'
        ),
    }

print('Cached generations:', len(INFERENCE_CACHE))


## 9. Aggregate the comparison

Pass@N means that at least one configured seed succeeded for a task. Mean best
cell accuracy is computed only where a seed produced the correct output shape;
the denominator is shown explicitly.


In [ ]:
from html import escape
from IPython.display import HTML, display


def active_cache_entry(
    panel_name: str,
    schedule_name: str,
    num_steps: int,
    seed: int,
) -> dict[str, object]:
    conditioned = get_conditioned_panel(panel_name, ACTIVE_TASK_IDS)
    key = inference_cache_key(
        panel_name,
        tuple(conditioned['task_ids']),
        schedule_name,
        num_steps,
        seed,
    )
    if key not in INFERENCE_CACHE:
        raise RuntimeError(f'Missing cached inference: {key}')
    return INFERENCE_CACHE[key]


def build_summary_rows() -> list[dict[str, object]]:
    rows = []
    for panel_name in ACTIVE_PANEL_NAMES:
        for schedule_name, num_steps in ACTIVE_SCHEDULE_SPECS:
            entries = [
                active_cache_entry(panel_name, schedule_name, num_steps, seed)
                for seed in ACTIVE_GENERATION_SEEDS
            ]
            diagnostics_by_seed = [entry['diagnostics'] for entry in entries]
            task_count = len(entries[0]['task_ids'])
            exact_pass1 = sum(item.exact_grid for item in diagnostics_by_seed[0])
            exact_passn = 0
            shape_passn = 0
            valid_passn = 0
            best_cell_accuracies = []
            for task_diagnostics in zip(*diagnostics_by_seed, strict=True):
                exact_passn += any(item.exact_grid for item in task_diagnostics)
                shape_passn += any(item.shape_match for item in task_diagnostics)
                valid_passn += any(item.prediction_valid for item in task_diagnostics)
                available = [
                    item.cell_accuracy
                    for item in task_diagnostics
                    if item.cell_accuracy is not None
                ]
                if available:
                    best_cell_accuracies.append(max(available))
            rows.append({
                'panel': panel_name,
                'schedule': schedule_name,
                'steps': num_steps,
                'tasks': task_count,
                'seeds': len(entries),
                'exact_pass1': exact_pass1,
                'exact_passn': exact_passn,
                'shape_passn': shape_passn,
                'valid_passn': valid_passn,
                'mean_best_cell_accuracy': (
                    sum(best_cell_accuracies) / len(best_cell_accuracies)
                    if best_cell_accuracies else None
                ),
                'cell_accuracy_tasks': len(best_cell_accuracies),
                'generation_seconds': sum(entry['seconds'] for entry in entries),
            })
    return rows


def display_summary_table(rows: list[dict[str, object]]) -> None:
    headers = (
        'panel', 'schedule', 'steps', 'tasks', 'seeds', 'exact pass@1',
        'exact pass@N', 'shape pass@N', 'valid pass@N',
        'mean best cell', 'cell denominator', 'seconds',
    )
    table_rows = []
    for row in rows:
        mean_cell = row['mean_best_cell_accuracy']
        values = (
            row['panel'], row['schedule'], row['steps'], row['tasks'], row['seeds'],
            row['exact_pass1'], row['exact_passn'], row['shape_passn'],
            row['valid_passn'],
            '—' if mean_cell is None else f'{100 * mean_cell:.1f}%',
            row['cell_accuracy_tasks'], f'{row["generation_seconds"]:.1f}',
        )
        cells = ''.join(f'<td>{escape(str(value))}</td>' for value in values)
        table_rows.append(f'<tr>{cells}</tr>')
    header_html = ''.join(f'<th>{escape(header)}</th>' for header in headers)
    display(HTML(
        '<table><thead><tr>' + header_html + '</tr></thead><tbody>'
        + ''.join(table_rows)
        + '</tbody></table>'
    ))


SUMMARY_ROWS = build_summary_rows()
display_summary_table(SUMMARY_ROWS)


## 10. Interactive per-task display

Change DISPLAY_PANEL, DISPLAY_TASK_ID, or DISPLAY_SEED and rerun only the next
cell. The upper row shows the target and raw VAE outputs. The lower row shows
the strictly decoded grids, or the complete classified 30x30 canvas when PAD
boundaries make a prediction invalid.


In [ ]:
import numpy as np
from matplotlib.colors import ListedColormap

from diffusion_arc.arc_data import PAD_CLASS, TARGET_PALETTE_RGB

ARC_CMAP = ListedColormap(TARGET_PALETTE_RGB.numpy() / 255.0)


def show_grid(axis, values, title: str) -> None:
    array = np.asarray(values, dtype=np.int64)
    axis.imshow(
        array,
        cmap=ARC_CMAP,
        vmin=-0.5,
        vmax=PAD_CLASS + 0.5,
        interpolation='nearest',
    )
    axis.set_title(title, fontsize=9)
    axis.set_xticks([])
    axis.set_yticks([])
    axis.set_aspect('equal')


def plot_task_comparison(panel_name: str, task_id: str, seed: int) -> None:
    entries = [
        active_cache_entry(panel_name, schedule_name, num_steps, seed)
        for schedule_name, num_steps in ACTIVE_SCHEDULE_SPECS
    ]
    task_ids = tuple(entries[0]['task_ids'])
    if task_id not in task_ids:
        raise ValueError(f'Task {task_id} is absent from the active selection')
    index = task_ids.index(task_id)
    columns = len(entries) + 1
    figure, axes = plt.subplots(
        2,
        columns,
        figsize=(3.2 * columns, 6.6),
        squeeze=False,
    )
    show_grid(axes[0, 0], entries[0]['expected_grids'][index], 'Expected target')
    axes[1, 0].axis('off')
    axes[1, 0].text(
        0.5,
        0.5,
        f'{panel_name}\n{task_id}\nseed {seed}',
        ha='center',
        va='center',
    )

    for column, entry in enumerate(entries, start=1):
        schedule_label = f'{entry["schedule_name"]} / {entry["num_steps"]}'
        rgb = entry['rgb_images'][index].permute(1, 2, 0).numpy()
        axes[0, column].imshow(rgb)
        axes[0, column].set_title(f'{schedule_label}\nraw VAE RGB', fontsize=9)
        axes[0, column].axis('off')

        prediction = entry['predictions'][index]
        diagnostic = entry['diagnostics'][index]
        if prediction.grid is None:
            decoded = entry['class_canvases'][index].numpy()
            decoded_label = f'invalid: {prediction.invalid_reason}'
        else:
            decoded = prediction.grid
            decoded_label = f'decoded {prediction.height}×{prediction.width}'
        if diagnostic.exact_grid:
            metric_label = 'EXACT'
        elif diagnostic.cell_accuracy is not None:
            metric_label = f'cell {100 * diagnostic.cell_accuracy:.1f}%'
        elif diagnostic.shape_match:
            metric_label = 'shape match'
        else:
            metric_label = 'shape mismatch'
        show_grid(
            axes[1, column],
            decoded,
            f'{decoded_label}\n{metric_label}',
        )

    figure.suptitle('Matched-noise inference comparison', fontsize=13)
    plt.tight_layout()
    plt.show()


DISPLAY_PANEL = ACTIVE_PANEL_NAMES[0]
DISPLAY_TASK_ID = (
    ACTIVE_TASK_IDS[0] if ACTIVE_TASK_IDS else PANEL_PROMPTS[DISPLAY_PANEL]['task_ids'][0]
)
DISPLAY_SEED = ACTIVE_GENERATION_SEEDS[0]
plot_task_comparison(DISPLAY_PANEL, DISPLAY_TASK_ID, DISPLAY_SEED)


In [ ]:
# Set to True and rerun to display every active task for one panel and seed.
DISPLAY_ALL_TASKS = False
if DISPLAY_ALL_TASKS:
    conditioned = get_conditioned_panel(DISPLAY_PANEL, ACTIVE_TASK_IDS)
    for task_id in conditioned['task_ids']:
        plot_task_comparison(DISPLAY_PANEL, task_id, DISPLAY_SEED)
else:
    print('Set DISPLAY_ALL_TASKS = True to render the full active panel.')


## 11. Optional durable result export

This saves the active cached comparison as a compact report plus one PNG per
sample. It deliberately performs no explicit Modal Volume commit; ordinary
writes to the mounted path are used, matching the files already visible from
the notebook run.


In [ ]:
from PIL import Image


def diagnostic_payload(diagnostic) -> dict[str, object]:
    return {
        'prediction_valid': diagnostic.prediction_valid,
        'invalid_reason': diagnostic.invalid_reason,
        'predicted_shape': diagnostic.predicted_shape,
        'expected_shape': diagnostic.expected_shape,
        'shape_match': diagnostic.shape_match,
        'cell_accuracy': diagnostic.cell_accuracy,
        'exact_grid': diagnostic.exact_grid,
    }


def save_active_results() -> Path:
    images_root = OUTPUT_ROOT / 'generated_images'
    images_root.mkdir(parents=True, exist_ok=True)
    serialized_runs = []

    for panel_name in ACTIVE_PANEL_NAMES:
        for schedule_name, num_steps in ACTIVE_SCHEDULE_SPECS:
            for seed in ACTIVE_GENERATION_SEEDS:
                entry = active_cache_entry(panel_name, schedule_name, num_steps, seed)
                samples = []
                for index, task_id in enumerate(entry['task_ids']):
                    filename = (
                        f'{panel_name}-{schedule_name}-{num_steps}-'
                        f'seed-{seed}-{index:02d}-{task_id}.png'
                    )
                    image = entry['rgb_images'][index].permute(1, 2, 0).numpy()
                    Image.fromarray(image).save(images_root / filename)
                    prediction = entry['predictions'][index]
                    samples.append({
                        'task_id': task_id,
                        'key': entry['keys'][index],
                        'expected_grid': entry['expected_grids'][index],
                        'prediction': prediction.grid,
                        'image': str((images_root / filename).relative_to(OUTPUT_ROOT)),
                        'diagnostic': diagnostic_payload(entry['diagnostics'][index]),
                    })
                serialized_runs.append({
                    'panel': panel_name,
                    'schedule': schedule_name,
                    'num_steps': num_steps,
                    'seed': seed,
                    'seconds': entry['seconds'],
                    'memory': entry['memory'],
                    'samples': samples,
                })

    report = {
        'schema_version': 1,
        'status': 'COMPLETE',
        'evaluation_id': EVALUATION_ID,
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'question': (
            'Does a BFL-shifted or longer Euler schedule improve the saved '
            'overfitting checkpoint without retraining?'
        ),
        'training_run_id': TRAINING_RUN_ID,
        'training_report_status': training_report['status'],
        'checkpoint': CHECKPOINT_MANIFEST,
        'source_revision': RESOLVED_SOURCE_COMMIT,
        'environment': {
            'python': platform.python_version(),
            'torch': torch.__version__,
            'cuda_runtime': torch.version.cuda,
            'gpu_name': properties.name,
            'gpu_vram_gib': gpu_vram_gib,
            'packages': {
                name: installed_version(name) for name in PINNED_PACKAGES
            },
        },
        'configuration': {
            'panels': ACTIVE_PANEL_NAMES,
            'task_ids': ACTIVE_TASK_IDS or TASK_IDS,
            'schedule_specs': ACTIVE_SCHEDULE_SPECS,
            'generation_seeds': ACTIVE_GENERATION_SEEDS,
            'image_sequence_length': IMAGE_SEQUENCE_LENGTH,
        },
        'summary': build_summary_rows(),
        'runs': serialized_runs,
    }
    report_path = OUTPUT_ROOT / 'report.json'
    temporary = report_path.with_suffix('.json.tmp')
    temporary.write_text(
        json.dumps(report, indent=2, sort_keys=True, allow_nan=False) + '\n',
        encoding='utf-8',
    )
    temporary.replace(report_path)
    if hasattr(os, 'sync'):
        os.sync()
    return report_path


if SAVE_RESULTS:
    SAVED_REPORT_PATH = save_active_results()
    print('Saved comparison report:', SAVED_REPORT_PATH)
else:
    print('SAVE_RESULTS is False; results remain in the notebook cache only.')


## 12. Interpretation boundary and next decision

- If BFL-50 improves over linear-50 with matched noise, the original schedule was
  suppressing checkpoint quality.
- If only 100 or 200 steps improve, Euler discretization error matters, but the
  gain must be weighed against proportional inference cost.
- If all schedules remain nearly identical and condition-insensitive, sampler
  choice is unlikely to explain the failed overfit. The next controlled test is
  then a one-task run with reduced canvas/padding and lower or decayed learning
  rate.
- This tiny, training-task-only comparison is diagnostic evidence. It is not an
  ARC validation result and does not establish generalization.
